# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 07 — RAG Completo com Groq
**Do PDF à Resposta Fundamentada · Chunking Fixo vs. Recursivo · Grounding e Citação da Fonte**

---

### Objetivos do Laboratório
1. Construir uma pipeline completa de **RAG (Retrieval-Augmented Generation)** ponta a ponta.
2. Comparar estratégias de corte textual: **Tamanho Fixo** vs. **Chunking Recursivo** com `langchain_text_splitters`.
3. Indexar o conhecimento de um arquivo PDF (`introducaoml.pdf`) via embeddings vetoriais.
4. Implementar **Grounding Estrito**: forçar o LLM a responder **apenas** com base nas evidências fornecidas no contexto.
5. Garantir rastreabilidade através da **Citação da Fonte** e medição da pontuação de similaridade.
6. Testar o comportamento do sistema diante de perguntas fora do escopo do documento.


### 1. Dependências e Configuração de Chave de API
Bibliotecas necessárias:
```bash
pip install groq sentence-transformers scikit-learn pdfplumber python-dotenv langchain-text-splitters
```


In [ ]:
import os
import re
import pdfplumber
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

# Carrega variáveis de ambiente do arquivo .env caso exista
load_dotenv()

# Tentativa de inicialização do cliente Groq
from groq import Groq

# Selecionamos a chave Groq do ambiente ou solicitamos
groq_api_key = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API_KEY_2")
client_groq = None

if groq_api_key:
    try:
        client_groq = Groq(api_key=groq_api_key)
        print("Cliente Groq autenticado com sucesso!")
    except Exception as e:
        print(f"Aviso ao inicializar Groq: {e}")
else:
    print("Aviso: Chave GROQ_API_KEY não encontrada no arquivo .env.")
    print("O notebook executará com um simulador local demonstrativo caso a chave não seja configurada.")

---
### 2. Leitura do PDF e Comparativo de Estratégias de Chunking

In [ ]:
def ler_pdf(caminho):
    texto = ""
    with pdfplumber.open(caminho) as pdf:
        for p in pdf.pages:
            t = p.extract_text()
            if t: texto += t + " "
    return texto.strip()

arquivo_pdf = "introducaoml.pdf"
if os.path.exists(arquivo_pdf):
    texto_completo = ler_pdf(arquivo_pdf)
    print(f"Texto extraído do PDF com sucesso! Total de caracteres: {len(texto_completo)}")
else:
    print("Aviso: Gerando conteúdo representativo para o PDF.")
    texto_completo = '''Machine learning é um campo da inteligência artificial que desenvolve algoritmos capazes de aprender padrões a partir de dados. Os principais tipos incluem aprendizado supervisionado, não supervisionado e aprendizado por reforço. No aprendizado supervisionado, o algoritmo treina a partir de pares de entrada e saída rotulada, prevendo saídas para novas instâncias. No aprendizado não supervisionado, o modelo descobre agrupamentos e estruturas intrínsecas ocultas em dados não rotulados.'''

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Estratégia 1: Tamanho Fixo (corta estritamente por contagem de caracteres)
splitter_fixo = CharacterTextSplitter(
    separator="",
    chunk_size=500,
    chunk_overlap=100
)
chunks_fixo = splitter_fixo.split_text(texto_completo)

# Estratégia 2: Recursivo (respeita quebras naturais: parágrafo -> frase -> palavra)
splitter_recursivo = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=500,
    chunk_overlap=100
)
chunks_recursivo = splitter_recursivo.split_text(texto_completo)

print(f"=== COMPARATIVO DE CHUNKING ===")
print(f"Total de chunks (Tamanho Fixo):  {len(chunks_fixo)}")
print(f"Total de chunks (Recursivo):     {len(chunks_recursivo)}")
print("\nPrimeiro chunk recursivo:\n", chunks_recursivo[0][:200], "...")

---
### 3. Geração de Embeddings e Base de Conhecimento Vetorial

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    emb_chunks = modelo_emb.encode(chunks_recursivo)
    print(f"Embeddings calculados! Matriz: {emb_chunks.shape}")
except Exception as e:
    print(f"Usando vetorização demonstrativa rápida: {e}")
    # Simulação determinística de embeddings
    np.random.seed(42)
    emb_chunks = np.random.randn(len(chunks_recursivo), 384)
    emb_chunks = emb_chunks / np.linalg.norm(emb_chunks, axis=1, keepdims=True)
    
    class FakeEmb:
        def encode(self, texts):
            np.random.seed(99)
            v = np.random.randn(1, 384)
            return v / np.linalg.norm(v)
    modelo_emb = FakeEmb()

---
### 4. Implementação da Função de RAG com Grounding Estrito

In [ ]:
MODELO_LLM = "llama-3.1-8b-instant"  # Modelo rápido e eficiente da Groq

def responder_com_rag(pergunta, top_k=2):
    # 1. RECUPERAR: Vetoriza a pergunta e busca os trechos mais próximos
    emb_p = modelo_emb.encode([pergunta])
    if emb_p.ndim == 1:
        emb_p = emb_p.reshape(1, -1)
        
    sims = cosine_similarity(emb_p, emb_chunks)[0]
    top_indices = sims.argsort()[::-1][:top_k]
    
    contexto = "\n---\n".join(chunks_recursivo[i] for i in top_indices)
    fonte_principal = chunks_recursivo[top_indices[0]]
    score_similaridade = sims[top_indices[0]]
    
    # 2. AUMENTAR: Montagem do prompt com instrução de grounding rígida
    prompt = f"""Responda a pergunta do usuário usando APENAS as informações contidas no contexto abaixo.
Se a informação não estiver presente de forma explícita no contexto, responda exatamente: 'Não encontrei essa informação.'

CONTEXTO RECUPERADO DO DOCUMENTO:
{contexto}

PERGUNTA:
{pergunta}
"""

    # 3. GERAR: Chamada ao modelo de linguagem
    if client_groq is not None:
        try:
            resposta_api = client_groq.chat.completions.create(
                model=MODELO_LLM,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            resposta = resposta_api.choices[0].message.content
        except Exception as err:
            resposta = f"[Erro na chamada da API Groq: {err}]"
    else:
        # Resposta simulada para execução offline
        if "aprendizado de máquina" in pergunta.lower():
            resposta = "Aprendizado de máquina é um ramo da inteligência artificial que desenvolve algoritmos capazes de aprender padrões a partir de dados."
        else:
            resposta = "Não encontrei essa informação."
            
    return resposta, fonte_principal, score_similaridade

---
### 5. Execução dos Testes Práticos e Avaliação do Grounding

In [ ]:
# Teste 1: Pergunta contida no documento PDF
pergunta_1 = "O que é aprendizado de máquina?"
resposta, fonte, score = responder_com_rag(pergunta_1)

print("="*60)
print(f"PERGUNTA 1: {pergunta_1}")
print("="*60)
print("RESPOSTA DO SISTEMA:\n", resposta)
print(f"\nBASEADO NO TRECHO DO PDF (Similaridade: {score:.4f}):")
print(f"'{fonte[:250]}...'")

In [ ]:
# Teste 2: Pergunta fora do escopo (Teste de Grounding e Anti-Alucinação)
pergunta_2 = "Qual a receita de bolo de cenoura?"
resposta, fonte, score = responder_com_rag(pergunta_2)

print("="*60)
print(f"PERGUNTA 2: {pergunta_2}")
print("="*60)
print("RESPOSTA DO SISTEMA:\n", resposta)
print(f"\nSCORE DO CHUNK MAIS PRÓXIMO: {score:.4f}")
print("\nValidação: O sistema não inventou uma receita porque o prompt instrui responder apenas com base no contexto!")

---
### 6. Quadro-Resumo: As 4 Etapas do Pipeline RAG

| Etapa | O que faz | Tecnologia / Ferramenta |
| :--- | :--- | :--- |
| **1. Indexar** | Lê o PDF, divide em blocos e gera representações vetoriais | `pdfplumber`, `RecursiveCharacterTextSplitter`, `SentenceTransformers` |
| **2. Recuperar** | Transforma a dúvida em vetor e busca os pedaços mais próximos | `cosine_similarity` |
| **3. Aumentar** | Injeta os pedaços resgatados dentro do prompt com regras de contenção | Engenharia de Prompt e Grounding |
| **4. Gerar** | LLM lê os dados recuperados e sintetiza a resposta final citando o trecho | `Groq API` (`llama-3.1-8b-instant`) |
